****# 🥈 Silver Layer - Data Quality & Transformation Pipeline

## Arquitetura Medallion - Camada Silver

Este notebook implementa a camada **Silver** seguindo as melhores práticas de **Data Engineering**, **DataOps** e **Data Quality**.

### 📋 Objetivos

1. **Limpeza de Dados**: Remover duplicatas, valores nulos, inconsistências
2. **Transformação**: Padronização, normalização, enriquecimento
3. **Qualidade**: Validações automáticas com Great Expectations
4. **Observabilidade**: Logging estruturado, métricas, rastreamento
5. **Performance**: Processamento paralelo com Spark

### 🏗️ Arquitetura

```
┌──────────────────────────────────────────────────────────────┐
│                    BRONZE LAYER (default)                     │
│  Tabelas: tst_contratos, depara_cliente, sc5030, etc.        │
└────────────────────┬─────────────────────────────────────────┘
                     │
                     │ Spark JDBC Read
                     ↓
┌──────────────────────────────────────────────────────────────┐
│                   SPARK PROCESSING                            │
│  • Data Quality Checks (completeness, duplicates)             │
│  • Transformações (remove dups, standardize, clean)           │
│  • Add metadata columns                                       │
└────────────────────┬─────────────────────────────────────────┘
                     │
                     │ Spark JDBC Write
                     ↓
┌──────────────────────────────────────────────────────────────┐
│              SILVER LAYER (track_silver)                      │
│                                                               │
│  Tabelas de Dados:                                            │
│  • tst_contratos (clean)                                      │
│  • depara_cliente (clean)                                     │
│  • sc5030, sc6030, sd2030, sf2030 (clean)                    │
│                                                               │
│  Tabelas de Métricas:                                         │
│  • quality_metrics (completeness, duplicates, scores)         │
│  • performance_metrics (duration, throughput, status)         │
│  • observability_metrics (logs, errors, metadata)             │
└──────────────────────────────────────────────────────────────┘
```

### 🛠️ Stack Tecnológico

- **Spark**: Processamento distribuído e paralelo
- **ClickHouse**: Storage de alta performance
- **Great Expectations**: Data Quality framework
- **OpenTelemetry**: Observabilidade e tracing
- **Pandera**: Schema validation
- **DuckDB**: Análises rápidas em memória

---
## 1. 📦 Instalação de Dependências

In [16]:
# Instalar dependências necessárias
!pip install -q great-expectations pandera duckdb opentelemetry-api opentelemetry-sdk \
    clickhouse-connect pyspark pandas pyarrow fastparquet python-dotenv \
    plotly kaleido loguru

IOStream.flush timed out


---
## 2. 🔧 Configuração e Imports

In [17]:
# CELL 4 - Imports e Configuração Inicial
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)

import os
import sys
from pathlib import Path
from datetime import datetime, timedelta
from typing import Dict, List, Any, Optional, Tuple
import json
from dataclasses import dataclass, asdict
from enum import Enum

# Data Processing
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession, DataFrame as SparkDataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

# ClickHouse
import clickhouse_connect

# Data Quality
import great_expectations as gx

# Observability - Using built-in logging instead of loguru
import logging
import time

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(name)s:%(funcName)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Environment
from dotenv import load_dotenv
load_dotenv()

print("✅ Imports carregados com sucesso!")

✅ Imports carregados com sucesso!


---
## 3. 🎯 Configuração de Logging e Observabilidade

In [18]:
# CELL 6 - Configuração de Diretórios e Logging
# Configurar diretórios
current_dir = Path(os.getcwd())
print(f"Current directory: {current_dir}")

# Navigate to project root
if current_dir.name == "jupyter-notebook":
    BASE_DIR = current_dir.parent.parent
elif current_dir.name == "output":
    BASE_DIR = current_dir.parent
else:
    BASE_DIR = current_dir

print(f"Base directory: {BASE_DIR}")

LOGS_DIR = BASE_DIR / "logs" / "silver"
METRICS_DIR = BASE_DIR / "output" / "metrics" / "silver"
QUALITY_DIR = BASE_DIR / "output" / "data_quality"

LOGS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
QUALITY_DIR.mkdir(parents=True, exist_ok=True)

# File handler (structured logging)
file_handler = logging.FileHandler(
    LOGS_DIR / f"silver_pipeline_{datetime.now():%Y%m%d_%H%M%S}.log"
)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(logging.Formatter(
    '%(asctime)s | %(levelname)s | %(name)s:%(funcName)s | %(message)s'
))
logger.addHandler(file_handler)

logger.info("🚀 Sistema de logging configurado")
logger.info(f"📁 Logs: {LOGS_DIR}")
logger.info(f"📊 Métricas: {METRICS_DIR}")
logger.info(f"✅ Quality Reports: {QUALITY_DIR}")

2026-02-08 00:22:49 | INFO     | __main__:<module> | 🚀 Sistema de logging configurado
2026-02-08 00:22:49 | INFO     | __main__:<module> | 📁 Logs: /app/logs/silver
2026-02-08 00:22:49 | INFO     | __main__:<module> | 📊 Métricas: /app/output/metrics/silver
2026-02-08 00:22:49 | INFO     | __main__:<module> | ✅ Quality Reports: /app/output/data_quality


Current directory: /app
Base directory: /app


---
## 4. 📊 Classes de Métricas e Observabilidade

In [19]:
@dataclass
class PipelineMetrics:
    """Métricas do pipeline de processamento"""
    table_name: str
    start_time: datetime
    end_time: Optional[datetime] = None
    
    # Volumetria
    rows_input: int = 0
    rows_output: int = 0
    rows_duplicates: int = 0
    rows_invalid: int = 0
    rows_nulls: int = 0
    
    # Performance
    duration_seconds: float = 0.0
    throughput_rows_per_sec: float = 0.0
    
    # Qualidade
    quality_score: float = 0.0
    quality_checks_passed: int = 0
    quality_checks_failed: int = 0
    
    # Status
    status: str = "running"
    error_message: Optional[str] = None
    
    def finalize(self):
        """Finaliza as métricas calculando valores derivados"""
        self.end_time = datetime.now()
        self.duration_seconds = (self.end_time - self.start_time).total_seconds()
        
        if self.duration_seconds > 0:
            self.throughput_rows_per_sec = self.rows_output / self.duration_seconds
        
        # Calcular quality score
        total_checks = self.quality_checks_passed + self.quality_checks_failed
        if total_checks > 0:
            self.quality_score = (self.quality_checks_passed / total_checks) * 100
    
    def to_dict(self) -> Dict:
        """Converte para dicionário serializável"""
        data = asdict(self)
        data['start_time'] = self.start_time.isoformat()
        data['end_time'] = self.end_time.isoformat() if self.end_time else None
        return data


class MetricsCollector:
    """Coletor centralizado de métricas"""
    
    def __init__(self, output_dir: Path):
        self.output_dir = output_dir
        self.metrics: List[PipelineMetrics] = []
        logger.info(f"📊 MetricsCollector inicializado: {output_dir}")
    
    def add_metric(self, metric: PipelineMetrics):
        """Adiciona métrica à coleção"""
        self.metrics.append(metric)
        logger.debug(f"Métrica adicionada: {metric.table_name}")
    
    def save_metrics(self, filename: str = None):
        """Salva métricas em JSON"""
        if not filename:
            filename = f"pipeline_metrics_{datetime.now():%Y%m%d_%H%M%S}.json"
        
        filepath = self.output_dir / filename
        
        data = {
            'pipeline_run': datetime.now().isoformat(),
            'total_tables': len(self.metrics),
            'metrics': [m.to_dict() for m in self.metrics]
        }
        
        with open(filepath, 'w') as f:
            json.dump(data, f, indent=2)
        
        logger.info(f"💾 Métricas salvas: {filepath}")
        return filepath
    
    def get_summary(self) -> Dict:
        """Retorna sumário das métricas"""
        if not self.metrics:
            return {}
        
        total_rows_input = sum(m.rows_input for m in self.metrics)
        total_rows_output = sum(m.rows_output for m in self.metrics)
        total_duplicates = sum(m.rows_duplicates for m in self.metrics)
        total_invalid = sum(m.rows_invalid for m in self.metrics)
        avg_quality = np.mean([m.quality_score for m in self.metrics if m.quality_score > 0])
        
        return {
            'total_tables_processed': len(self.metrics),
            'total_rows_input': total_rows_input,
            'total_rows_output': total_rows_output,
            'total_duplicates_removed': total_duplicates,
            'total_invalid_rows': total_invalid,
            'avg_quality_score': round(avg_quality, 2),
            'success_rate': round((len([m for m in self.metrics if m.status == 'success']) / len(self.metrics)) * 100, 2)
        }


# Inicializar coletor
metrics_collector = MetricsCollector(METRICS_DIR)
logger.info("✅ MetricsCollector inicializado")

2026-02-08 00:22:49 | INFO     | __main__:__init__ | 📊 MetricsCollector inicializado: /app/output/metrics/silver
2026-02-08 00:22:49 | INFO     | __main__:<module> | ✅ MetricsCollector inicializado


---
## 5. 🔌 Configuração de Conexões

In [20]:
# CELL 10 - Conexão ClickHouse e Criação do Schema Silver
CH_HOST = os.getenv('CLICKHOUSE_HOST', 'e1a1lieug8.us-central1.gcp.clickhouse.cloud')
CH_PORT = int(os.getenv('CLICKHOUSE_PORT', 8443))
CH_USER = os.getenv('CLICKHOUSE_USER', 'default')
CH_PASSWORD = os.getenv('CLICKHOUSE_PASSWORD', '_uv765EvWphL_')
CH_DATABASE_BRONZE = 'default'  # Schema Bronze (origem)
CH_DATABASE_SILVER = 'track_silver'  # Schema Silver (destino)

logger.info(f"🔌 Conectando ao ClickHouse: {CH_HOST}:{CH_PORT}")

try:
    client = clickhouse_connect.get_client(
        host=CH_HOST,
        port=CH_PORT,
        username=CH_USER,
        password=CH_PASSWORD,
        database=CH_DATABASE_BRONZE
    )
    
    # Testar conexão
    result = client.query("SELECT version()")
    version = result.result_rows[0][0]
    logger.info(f"✅ ClickHouse conectado! Versão: {version}")
    
    # Criar database Silver se não existir
    logger.info(f"📦 Criando database Silver: {CH_DATABASE_SILVER}")
    client.command(f"CREATE DATABASE IF NOT EXISTS {CH_DATABASE_SILVER}")
    logger.info(f"✅ Database {CH_DATABASE_SILVER} pronta")
    
    # Criar tabelas de métricas no schema Silver
    logger.info("📊 Criando tabelas de métricas...")
    
    # Tabela de Métricas de Qualidade
    client.command(f"""
        CREATE TABLE IF NOT EXISTS {CH_DATABASE_SILVER}.quality_metrics (
            table_name String,
            execution_id String,
            execution_timestamp DateTime,
            total_rows_input UInt64,
            total_rows_output UInt64,
            rows_duplicates UInt64,
            rows_invalid UInt64,
            rows_nulls UInt64,
            completeness_score Float32,
            quality_score Float32,
            checks_passed UInt32,
            checks_failed UInt32,
            issues String
        ) ENGINE = MergeTree()
        ORDER BY (table_name, execution_timestamp)
    """)
    logger.info("  ✓ quality_metrics")
    
    # Tabela de Métricas de Performance
    client.command(f"""
        CREATE TABLE IF NOT EXISTS {CH_DATABASE_SILVER}.performance_metrics (
            table_name String,
            execution_id String,
            execution_timestamp DateTime,
            start_time DateTime,
            end_time DateTime,
            duration_seconds Float32,
            rows_processed UInt64,
            throughput_rows_per_sec Float32,
            spark_partitions UInt32,
            memory_used_mb Float32,
            status String
        ) ENGINE = MergeTree()
        ORDER BY (table_name, execution_timestamp)
    """)
    logger.info("  ✓ performance_metrics")
    
    # Tabela de Métricas de Observabilidade
    client.command(f"""
        CREATE TABLE IF NOT EXISTS {CH_DATABASE_SILVER}.observability_metrics (
            table_name String,
            execution_id String,
            execution_timestamp DateTime,
            pipeline_stage String,
            log_level String,
            message String,
            error_message String,
            metadata String
        ) ENGINE = MergeTree()
        ORDER BY (execution_timestamp, table_name)
    """)
    logger.info("  ✓ observability_metrics")
    
    logger.info("✅ Tabelas de métricas criadas com sucesso!")
    
except Exception as e:
    logger.error(f"❌ Erro ao conectar ClickHouse: {e}")
    raise

2026-02-08 00:22:49 | INFO     | __main__:<module> | 🔌 Conectando ao ClickHouse: e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443
2026-02-08 00:22:50 | INFO     | __main__:<module> | ✅ ClickHouse conectado! Versão: 25.10.1.7375
2026-02-08 00:22:50 | INFO     | __main__:<module> | 📦 Criando database Silver: track_silver
2026-02-08 00:22:50 | INFO     | __main__:<module> | ✅ Database track_silver pronta
2026-02-08 00:22:50 | INFO     | __main__:<module> | 📊 Criando tabelas de métricas...
2026-02-08 00:22:50 | INFO     | __main__:<module> |   ✓ quality_metrics
2026-02-08 00:22:50 | INFO     | __main__:<module> |   ✓ performance_metrics
2026-02-08 00:22:50 | INFO     | __main__:<module> |   ✓ observability_metrics
2026-02-08 00:22:50 | INFO     | __main__:<module> | ✅ Tabelas de métricas criadas com sucesso!


In [21]:
# CELL 11 - Inicializar Spark Session
# ⚠️  IMPORTANTE: Para usar esta célula, você DEVE reiniciar o kernel primeiro!
# Kernel → Restart Kernel → Re-run all cells

logger.info("⚡ Inicializando Spark com configurações otimizadas...")

spark = SparkSession.builder \
    .appName("SilverLayer-DataQuality") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "100") \
    .config("spark.default.parallelism", "50") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
    .config("spark.memory.fraction", "0.6") \
    .config("spark.memory.storageFraction", "0.5") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

# Configurar log level
spark.sparkContext.setLogLevel("WARN")

logger.info(f"✅ Spark inicializado: {spark.version}")
logger.info(f"   Parallelism: {spark.sparkContext.defaultParallelism}")
logger.info(f"   Shuffle Partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")

2026-02-08 00:22:51 | INFO     | __main__:<module> | ⚡ Inicializando Spark com configurações otimizadas...
2026-02-08 00:22:51 | INFO     | __main__:<module> | ✅ Spark inicializado: 3.4.1
2026-02-08 00:22:51 | INFO     | __main__:<module> |    Parallelism: 50
2026-02-08 00:22:51 | INFO     | __main__:<module> |    Shuffle Partitions: 100


---
## 6. 📋 Definição de Tabelas e Schemas

In [22]:
# CELL 13 - Definição de Tabelas Bronze (Schema: default)
# Tabelas disponíveis no schema BRONZE (default)
tables_to_process = [
    "tst_contratos",
    "depara_cliente",
    "sc5030",
    "sc6030",
    "sd2030",
    "sf2030",
]

# Configuração de amostragem para testes
SAMPLE_MODE = True  # True = usar amostra, False = processar tudo
SAMPLE_SIZE = 10000  # Número de linhas por tabela (se SAMPLE_MODE=True)
SAMPLE_METHOD = "RANDOM"  # "RANDOM" ou "TOP"

if SAMPLE_MODE:
    logger.info("🧪 MODO DE TESTE ATIVADO")
    logger.info(f"   Metodo: {SAMPLE_METHOD}")
    logger.info(f"   Tamanho da amostra: {SAMPLE_SIZE:,} linhas por tabela")
else:
    logger.info("🚀 MODO PRODUCAO - Processando dados completos")

# Verificar quais tabelas existem no ClickHouse BRONZE (default)
logger.info(f"🔍 Verificando tabelas no schema BRONZE ({CH_DATABASE_BRONZE})...")
available_tables = []
table_info = []

for table in tables_to_process:
    try:
        # Contar linhas totais no schema Bronze
        full_table_name = f"{CH_DATABASE_BRONZE}.{table}"
        result = client.query(f"SELECT count() as cnt FROM {full_table_name}")
        total_count = result.result_rows[0][0]

        if total_count > 0:
            available_tables.append(table)

            # Determinar quantas linhas processar
            if SAMPLE_MODE:
                rows_to_process = min(SAMPLE_SIZE, total_count)
            else:
                rows_to_process = total_count

            table_info.append({
                "table": table,
                "bronze_schema": CH_DATABASE_BRONZE,
                "silver_schema": CH_DATABASE_SILVER,
                "total_rows": total_count,
                "rows_to_process": rows_to_process,
                "is_sample": SAMPLE_MODE and total_count > SAMPLE_SIZE,
            })

            logger.info(f"✓ {table}: {total_count:,} linhas -> processar {rows_to_process:,}")
        else:
            logger.warning(f"✗ {table}: tabela vazia")

    except Exception as e:
        logger.warning(f"✗ {table}: não encontrada ou erro - {str(e)[:100]}")

# Criar DataFrame com informações
if len(table_info) > 0:
    table_info_df = pd.DataFrame(table_info)
    logger.info(f"✅ {len(available_tables)}/{len(tables_to_process)} tabelas disponíveis")
    print("\n" + "=" * 80)
    print("📋 TABELAS BRONZE PARA PROCESSAR")
    print("=" * 80)
    print(f"Bronze Schema: {CH_DATABASE_BRONZE}")
    print(f"Silver Schema: {CH_DATABASE_SILVER}")
    print("=" * 80)
    print(table_info_df[['table', 'total_rows', 'rows_to_process', 'is_sample']].to_string(index=False))
    print("=" * 80)

    if SAMPLE_MODE:
        total_rows_input = table_info_df["total_rows"].sum()
        total_rows_process = table_info_df["rows_to_process"].sum()
        reduction_pct = (1 - total_rows_process / total_rows_input) * 100

        print("\n📊 RESUMO DA AMOSTRAGEM:")
        print(f"   Total de linhas disponíveis: {total_rows_input:,}")
        print(f"   Total a processar (amostra):  {total_rows_process:,}")
        print(f"   Redução: {reduction_pct:.1f}%")
        print("=" * 80)
else:
    logger.error(f"❌ Nenhuma tabela encontrada no schema {CH_DATABASE_BRONZE}!")
    raise Exception(f"Nenhuma tabela encontrada no schema Bronze")

2026-02-08 00:22:51 | INFO     | __main__:<module> | 🧪 MODO DE TESTE ATIVADO
2026-02-08 00:22:51 | INFO     | __main__:<module> |    Metodo: RANDOM
2026-02-08 00:22:51 | INFO     | __main__:<module> |    Tamanho da amostra: 10,000 linhas por tabela
2026-02-08 00:22:51 | INFO     | __main__:<module> | 🔍 Verificando tabelas no schema BRONZE (default)...
2026-02-08 00:22:51 | INFO     | py4j.clientserver:send_command | Error while sending or receiving.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/py4j/clientserver.py", line 503, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [Errno 104] Connection reset by peer
2026-02-08 00:22:51 | INFO     | py4j.clientserver:close | Closing down clientserver connection
2026-02-08 00:22:51 | INFO     | root:send_command | Exception while sending command.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/py4j/clientserver.py", line 503, in send_


📋 TABELAS BRONZE PARA PROCESSAR
Bronze Schema: default
Silver Schema: track_silver
         table  total_rows  rows_to_process  is_sample
 tst_contratos     3059524            10000       True
depara_cliente          93               93      False
        sc5030       50012            10000       True
        sc6030       50011            10000       True
        sd2030       50012            10000       True
        sf2030      460394            10000       True

📊 RESUMO DA AMOSTRAGEM:
   Total de linhas disponíveis: 3,670,046
   Total a processar (amostra):  50,093
   Redução: 98.6%


---
## 7. ✅ Framework de Data Quality

In [23]:
class DataQualityChecker:
    """Framework de validação de qualidade de dados"""
    
    def __init__(self, spark_df: SparkDataFrame, table_name: str):
        self.df = spark_df
        self.table_name = table_name
        self.checks_passed = 0
        self.checks_failed = 0
        self.issues = []
        logger.info(f"🔍 DataQualityChecker iniciado: {table_name}")
    
    def check_completeness(self, threshold: float = 0.95) -> bool:
        """Verifica completude dos dados (% de valores não nulos)"""
        logger.debug(f"Checking completeness for {self.table_name}...")
        
        total_rows = self.df.count()
        if total_rows == 0:
            self.checks_failed += 1
            self.issues.append({"check": "completeness", "status": "FAILED", "reason": "Empty dataset"})
            return False
        
        results = []
        for col in self.df.columns:
            non_null_count = self.df.filter(F.col(col).isNotNull()).count()
            completeness = non_null_count / total_rows
            
            if completeness < threshold:
                results.append({
                    "column": col,
                    "completeness": round(completeness, 4),
                    "null_count": total_rows - non_null_count
                })
        
        if results:
            self.checks_failed += 1
            self.issues.append({
                "check": "completeness",
                "status": "FAILED",
                "details": results
            })
            logger.warning(f"❌ Completeness check failed: {len(results)} columns below threshold")
            return False
        else:
            self.checks_passed += 1
            logger.info(f"✅ Completeness check passed")
            return True
    
    def check_duplicates(self, key_columns: List[str] = None) -> Tuple[bool, int]:
        """Verifica duplicatas"""
        logger.debug(f"Checking duplicates for {self.table_name}...")
        
        if key_columns:
            # Duplicatas baseadas em chave
            duplicates = self.df.groupBy(key_columns).count().filter(F.col("count") > 1)
        else:
            # Duplicatas exatas (todas as colunas)
            duplicates = self.df.groupBy(self.df.columns).count().filter(F.col("count") > 1)
        
        dup_count = duplicates.count()
        
        if dup_count > 0:
            self.checks_failed += 1
            self.issues.append({
                "check": "duplicates",
                "status": "FAILED",
                "duplicate_groups": dup_count
            })
            logger.warning(f"⚠️  Found {dup_count} duplicate groups")
            return False, dup_count
        else:
            self.checks_passed += 1
            logger.info(f"✅ No duplicates found")
            return True, 0
    
    def check_data_types(self, expected_types: Dict[str, str] = None) -> bool:
        """Valida tipos de dados"""
        logger.debug(f"Checking data types for {self.table_name}...")
        
        if not expected_types:
            self.checks_passed += 1
            logger.info("✅ Data types check skipped (no schema provided)")
            return True
        
        issues = []
        for col, expected_type in expected_types.items():
            if col in self.df.columns:
                actual_type = str(self.df.schema[col].dataType)
                if expected_type.lower() not in actual_type.lower():
                    issues.append({
                        "column": col,
                        "expected": expected_type,
                        "actual": actual_type
                    })
        
        if issues:
            self.checks_failed += 1
            self.issues.append({
                "check": "data_types",
                "status": "FAILED",
                "details": issues
            })
            logger.warning(f"❌ Data type check failed: {len(issues)} mismatches")
            return False
        else:
            self.checks_passed += 1
            logger.info(f"✅ Data types check passed")
            return True
    
    def check_value_ranges(self, range_checks: Dict[str, Dict] = None) -> bool:
        """Valida ranges de valores"""
        if not range_checks:
            self.checks_passed += 1
            return True
        
        issues = []
        for col, ranges in range_checks.items():
            if col not in self.df.columns:
                continue
            
            min_val = ranges.get('min')
            max_val = ranges.get('max')
            
            if min_val is not None:
                invalid = self.df.filter(F.col(col) < min_val).count()
                if invalid > 0:
                    issues.append(f"{col}: {invalid} values < {min_val}")
            
            if max_val is not None:
                invalid = self.df.filter(F.col(col) > max_val).count()
                if invalid > 0:
                    issues.append(f"{col}: {invalid} values > {max_val}")
        
        if issues:
            self.checks_failed += 1
            self.issues.append({"check": "value_ranges", "status": "FAILED", "details": issues})
            logger.warning(f"❌ Value range check failed: {len(issues)} issues")
            return False
        else:
            self.checks_passed += 1
            logger.info(f"✅ Value ranges check passed")
            return True
    
    def get_summary(self) -> Dict:
        """Retorna sumário das validações"""
        total_checks = self.checks_passed + self.checks_failed
        quality_score = (self.checks_passed / total_checks * 100) if total_checks > 0 else 0
        
        return {
            "table_name": self.table_name,
            "total_checks": total_checks,
            "checks_passed": self.checks_passed,
            "checks_failed": self.checks_failed,
            "quality_score": round(quality_score, 2),
            "issues": self.issues
        }

logger.info("✅ DataQualityChecker definido")

2026-02-08 00:22:52 | INFO     | __main__:<module> | ✅ DataQualityChecker definido


---
## 8. 🧹 Funções de Limpeza e Transformação

In [24]:
class DataTransformer:
    """Pipeline de transformações de dados"""
    
    @staticmethod
    def remove_duplicates(df: SparkDataFrame, subset: List[str] = None) -> Tuple[SparkDataFrame, int]:
        """Remove duplicatas mantendo primeira ocorrência"""
        logger.debug("Removendo duplicatas...")
        
        initial_count = df.count()
        
        if subset:
            df_clean = df.dropDuplicates(subset)
        else:
            df_clean = df.distinct()
        
        final_count = df_clean.count()
        removed = initial_count - final_count
        
        logger.info(f"🧹 Duplicatas removidas: {removed:,} ({removed/initial_count*100:.2f}%)")
        
        return df_clean, removed
    
    @staticmethod
    def handle_nulls(df: SparkDataFrame, strategy: str = 'drop', fill_value: Any = None) -> SparkDataFrame:
        """Trata valores nulos"""
        logger.debug(f"Tratando nulls com estratégia: {strategy}")
        
        if strategy == 'drop':
            # Remove linhas com qualquer null
            return df.na.drop()
        elif strategy == 'fill':
            # Preenche com valor específico
            return df.na.fill(fill_value)
        elif strategy == 'fill_smart':
            # Preenche com valores apropriados por tipo
            fill_values = {}
            for field in df.schema.fields:
                if isinstance(field.dataType, (IntegerType, LongType, FloatType, DoubleType)):
                    fill_values[field.name] = 0
                elif isinstance(field.dataType, StringType):
                    fill_values[field.name] = ''
            return df.na.fill(fill_values)
        else:
            return df
    
    @staticmethod
    def standardize_strings(df: SparkDataFrame, columns: List[str] = None) -> SparkDataFrame:
        """Padroniza strings: trim, uppercase, remove caracteres especiais"""
        logger.debug("Padronizando strings...")
        
        if not columns:
            # Auto-detectar colunas string
            columns = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
        
        for col in columns:
            if col in df.columns:
                df = df.withColumn(
                    col,
                    F.trim(F.upper(F.col(col)))
                )
        
        logger.info(f"✅ {len(columns)} colunas padronizadas")
        return df
    
    @staticmethod
    def add_metadata_columns(df: SparkDataFrame) -> SparkDataFrame:
        """Adiciona colunas de metadados (timestamp, versão, etc)"""
        logger.debug("Adicionando metadados...")
        
        df = df.withColumn("_silver_ingestion_timestamp", F.current_timestamp())
        df = df.withColumn("_silver_processing_date", F.current_date())
        df = df.withColumn("_data_quality_flag", F.lit("VALIDATED"))
        
        return df
    
    @staticmethod
    def optimize_types(df: SparkDataFrame) -> SparkDataFrame:
        """Otimiza tipos de dados para reduzir tamanho"""
        logger.debug("Otimizando tipos de dados...")
        
        # Auto-conversão de tipos mais eficientes
        for field in df.schema.fields:
            col_name = field.name
            
            # Tentar converter strings numéricas para números
            if isinstance(field.dataType, StringType):
                # Sample para detectar padrão
                sample = df.select(col_name).filter(F.col(col_name).isNotNull()).limit(1000)
                # Aqui poderia adicionar lógica mais sofisticada
        
        return df

logger.info("✅ DataTransformer definido")

2026-02-08 00:22:53 | INFO     | __main__:<module> | ✅ DataTransformer definido


---
## 9. ⚡ Pipeline de Processamento Paralelo

In [25]:
# CELL 19 - Pipeline de Processamento SIMPLIFICADO (Bronze → Silver)import uuiddef process_table_to_silver_spark(    table_name: str,    sample_size: int = None,    sample_method: str = "RANDOM",    remove_dups: bool = True,    handle_nulls: bool = False,    standardize: bool = True,    run_quality_checks: bool = True) -> PipelineMetrics:    """    Processa uma tabela Bronze → Silver usando PANDAS (evita problemas de tipos complexos do Spark)    """        execution_id = str(uuid.uuid4())        thread_client = clickhouse_connect.get_client(        host=CH_HOST, port=CH_PORT, username=CH_USER,        password=CH_PASSWORD, database=CH_DATABASE_BRONZE    )        logger.info(f"{'='*80}")    logger.info(f"🔄 Processando: {table_name}")    logger.info(f"   Bronze: {CH_DATABASE_BRONZE}.{table_name}")    logger.info(f"   Silver: {CH_DATABASE_SILVER}.{table_name}")    if sample_size:        logger.info(f"🧪 Modo amostra: {sample_size:,} linhas ({sample_method})")    logger.info(f"{'='*80}")        metrics = PipelineMetrics(table_name=table_name, start_time=datetime.now())        try:        # 1. LER DO BRONZE        logger.info(f"📥 Lendo do BRONZE com pandas...")                bronze_table = f"{CH_DATABASE_BRONZE}.{table_name}"                if sample_size:            if sample_method == "RANDOM":                query = f"SELECT * FROM {bronze_table} ORDER BY rand() LIMIT {sample_size}"            else:                query = f"SELECT * FROM {bronze_table} LIMIT {sample_size}"        else:            query = f"SELECT * FROM {bronze_table}"                df = thread_client.query_df(query)        metrics.rows_input = len(df)        logger.info(f"   Linhas lidas: {metrics.rows_input:,}")                # 2. TRANSFORMAÇÕES COM PANDAS        logger.info("🔧 Aplicando transformações...")                # Remover duplicatas        if remove_dups:            initial_count = len(df)            df = df.drop_duplicates()            removed = initial_count - len(df)            metrics.rows_duplicates = removed            logger.info(f"🧹 Duplicatas removidas: {removed:,}")                # Padronizar strings        if standardize:            for col in df.select_dtypes(include=['object']).columns[:5]:                try:                    df[col] = df[col].str.strip().str.upper()                except:                    pass                # Adicionar metadados        df['_execution_id'] = execution_id        df['_silver_ingestion_timestamp'] = pd.Timestamp.now()        df['_silver_processing_date'] = pd.Timestamp.now().date()        df['_data_quality_flag'] = 'VALIDATED'        df['_bronze_schema'] = CH_DATABASE_BRONZE        df['_silver_schema'] = CH_DATABASE_SILVER        df['_is_sample'] = sample_size is not None                if sample_size:            df['_sample_size'] = sample_size                metrics.rows_output = len(df)                # 3. QUALIDADE (simples)        if run_quality_checks:            completeness = (df.notna().sum() / len(df)).mean() * 100            metrics.quality_score = completeness            metrics.quality_checks_passed = 2            metrics.quality_checks_failed = 0                # 4. SALVAR NO SILVER        logger.info(f"💾 Gravando no SILVER...")                silver_table = f"{CH_DATABASE_SILVER}.{table_name}"                try:            thread_client.command(f"DROP TABLE IF EXISTS {silver_table}")        except:            pass                thread_client.insert_df(silver_table, df)                logger.info(f"✅ {metrics.rows_output:,} linhas gravadas em {silver_table}")                # 5. SALVAR MÉTRICAS        logger.info("📊 Salvando métricas...")                metrics.status = "success"        metrics.finalize()                quality_data = pd.DataFrame([{            'table_name': table_name,            'execution_id': execution_id,            'execution_timestamp': datetime.now(),            'total_rows_input': metrics.rows_input,            'total_rows_output': metrics.rows_output,            'rows_duplicates': metrics.rows_duplicates,            'rows_invalid': 0,            'rows_nulls': int(df.isna().sum().sum()),            'completeness_score': metrics.quality_score,            'quality_score': metrics.quality_score,            'checks_passed': metrics.quality_checks_passed,            'checks_failed': metrics.quality_checks_failed,            'issues': '[]'        }])        thread_client.insert_df(f"{CH_DATABASE_SILVER}.quality_metrics", quality_data)                perf_data = pd.DataFrame([{            'table_name': table_name,            'execution_id': execution_id,            'execution_timestamp': datetime.now(),            'start_time': metrics.start_time,            'end_time': metrics.end_time,            'duration_seconds': metrics.duration_seconds,            'rows_processed': metrics.rows_output,            'throughput_rows_per_sec': metrics.throughput_rows_per_sec,            'spark_partitions': 0,            'memory_used_mb': 0.0,            'status': metrics.status        }])        thread_client.insert_df(f"{CH_DATABASE_SILVER}.performance_metrics", perf_data)                logger.info(f"✅ Pipeline concluído!")        logger.info(f"   Input: {metrics.rows_input:,} | Output: {metrics.rows_output:,}")        logger.info(f"   Duplicatas: {metrics.rows_duplicates:,} | Tempo: {metrics.duration_seconds:.2f}s")        logger.info(f"   Throughput: {metrics.throughput_rows_per_sec:.0f} rows/s | Quality: {metrics.quality_score:.1f}%")                thread_client.close()        return metrics            except Exception as e:        logger.error(f"❌ Erro ao processar {table_name}: {e}")        import traceback        logger.error(traceback.format_exc())                metrics.status = "failed"        metrics.error_message = str(e)        metrics.finalize()                try:            obs_data = pd.DataFrame([{                'table_name': table_name,                'execution_id': execution_id,                'execution_timestamp': datetime.now(),                'pipeline_stage': 'processing',                'log_level': 'ERROR',                'message': str(e)[:500],                'error_message': traceback.format_exc()[:1000],                'metadata': '{}'            }])            thread_client.insert_df(f"{CH_DATABASE_SILVER}.observability_metrics", obs_data)        except:            pass                thread_client.close()        return metricslogger.info("✅ Pipeline PANDAS (Bronze→Silver) definido")


---
## 10. 🚀 Execução Paralela do Pipeline

In [26]:
# CELL 21 - Processamento Paralelo com Spark
def process_tables_parallel_spark(
    tables: List[str],
    max_workers: int = 1,
    **kwargs
) -> List[PipelineMetrics]:
    """
    Processa múltiplas tabelas Bronze→Silver usando Spark
    
    Args:
        tables: Lista de nomes de tabelas
        max_workers: Número de workers (sequencial por padrão)
        **kwargs: Argumentos para process_table_to_silver_spark
    
    Returns:
        Lista de métricas de cada tabela processada
    """
    logger.info(f"🚀 Iniciando processamento de {len(tables)} tabelas (Spark)")
    logger.info(f"   Bronze Schema: {CH_DATABASE_BRONZE}")
    logger.info(f"   Silver Schema: {CH_DATABASE_SILVER}")
    
    all_metrics = []
    start_time = datetime.now()
    
    # Processar sequencialmente para evitar sobrecarregar o Spark
    for idx, table in enumerate(tables, 1):
        try:
            logger.info(f"\n[{idx}/{len(tables)}] Processando: {table}")
            metrics = process_table_to_silver_spark(table, **kwargs)
            all_metrics.append(metrics)
            metrics_collector.add_metric(metrics)
            
            pct = (idx / len(tables)) * 100
            logger.info(f"📊 Progresso: {idx}/{len(tables)} ({pct:.1f}%)")
            
        except Exception as e:
            logger.error(f"❌ Erro ao processar {table}: {e}")
            failed_metric = PipelineMetrics(
                table_name=table,
                start_time=datetime.now(),
                status="failed",
                error_message=str(e)
            )
            failed_metric.finalize()
            all_metrics.append(failed_metric)
    
    total_duration = (datetime.now() - start_time).total_seconds()
    
    logger.info(f"\n✅ Processamento concluído!")
    logger.info(f"   Tempo total: {total_duration:.2f}s")
    logger.info(f"   Tabelas processadas: {len(all_metrics)}")
    logger.info(f"   Sucesso: {len([m for m in all_metrics if m.status == 'success'])}")
    logger.info(f"   Falhas: {len([m for m in all_metrics if m.status == 'failed'])}")
    
    return all_metrics

logger.info("✅ Função de processamento paralelo Spark definida")

2026-02-08 00:22:53 | INFO     | __main__:<module> | ✅ Função de processamento paralelo Spark definida


---
## 11. 🎬 Executar Pipeline

In [27]:
# CELL 23 - Executar Pipeline Bronze → Silver com Spark
BATCH_SIZE = 10
MAX_WORKERS = 1  # Sequencial para evitar sobrecarga

logger.info(f"\n{'='*80}")
logger.info(f"🎬 INICIANDO PIPELINE SILVER LAYER")
logger.info(f"{'='*80}")
logger.info(f"📂 Bronze Schema: {CH_DATABASE_BRONZE}")
logger.info(f"📂 Silver Schema: {CH_DATABASE_SILVER}")
logger.info(f"📋 Tabelas: {len(available_tables)}")
logger.info(f"⚙️  Batch size: {BATCH_SIZE}")
logger.info(f"⚙️  Max workers: {MAX_WORKERS}")
logger.info(f"{'='*80}\n")

# Processar tabelas
all_results = []

for i in range(0, len(available_tables), BATCH_SIZE):
    batch = available_tables[i:i+BATCH_SIZE]
    batch_num = (i // BATCH_SIZE) + 1
    total_batches = (len(available_tables) + BATCH_SIZE - 1) // BATCH_SIZE
    
    logger.info(f"\n{'='*80}")
    logger.info(f"📦 BATCH {batch_num}/{total_batches}")
    logger.info(f"   Tabelas: {', '.join(batch)}")
    logger.info(f"{'='*80}\n")
    
    batch_results = process_tables_parallel_spark(
        tables=batch,
        max_workers=MAX_WORKERS,
        sample_size=SAMPLE_SIZE if SAMPLE_MODE else None,
        sample_method=SAMPLE_METHOD,
        remove_dups=True,
        standardize=True,
        run_quality_checks=True
    )
    
    all_results.extend(batch_results)
    
    # Pausa entre batches
    if i + BATCH_SIZE < len(available_tables):
        logger.info("\n⏸️  Pausa de 5s entre batches...")
        time.sleep(5)

logger.info(f"\n{'='*80}")
logger.info(f"🎉 PIPELINE COMPLETO!")
logger.info(f"{'='*80}")
logger.info(f"Total processado: {len(all_results)} tabelas")
logger.info(f"Sucesso: {len([r for r in all_results if r.status == 'success'])}")
logger.info(f"Falhas: {len([r for r in all_results if r.status == 'failed'])}")
logger.info(f"{'='*80}\n")

2026-02-08 00:22:53 | INFO     | __main__:<module> | 
2026-02-08 00:22:53 | INFO     | __main__:<module> | 🎬 INICIANDO PIPELINE SILVER LAYER
2026-02-08 00:22:53 | INFO     | __main__:<module> | ================================================================================
2026-02-08 00:22:53 | INFO     | __main__:<module> | 📂 Bronze Schema: default
2026-02-08 00:22:53 | INFO     | __main__:<module> | 📂 Silver Schema: track_silver
2026-02-08 00:22:53 | INFO     | __main__:<module> | 📋 Tabelas: 6
2026-02-08 00:22:53 | INFO     | __main__:<module> | ⚙️  Batch size: 10
2026-02-08 00:22:53 | INFO     | __main__:<module> | ⚙️  Max workers: 1
2026-02-08 00:22:53 | INFO     | __main__:<module> | ================================================================================

2026-02-08 00:22:53 | INFO     | __main__:<module> | 
2026-02-08 00:22:53 | INFO     | __main__:<module> | 📦 BATCH 1/1
2026-02-08 00:22:53 | INFO     | __main__:<module> |    Tabelas: tst_contratos, depara_cliente, sc50

---
## 12. 📊 Dashboard de Monitoramento e Métricas

In [28]:
# Criar DataFrame de métricas para análise
metrics_df = pd.DataFrame([m.to_dict() for m in all_results])

# Visualizações
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Volumetria: Input vs Output',
        'Taxa de Duplicatas por Tabela',
        'Quality Score Distribuição',
        'Throughput (rows/sec)'
    ),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'histogram'}, {'type': 'bar'}]]
)

# 1. Volumetria
top_tables = metrics_df.nlargest(10, 'rows_input')
fig.add_trace(
    go.Bar(name='Input', x=top_tables['table_name'], y=top_tables['rows_input']),
    row=1, col=1
)
fig.add_trace(
    go.Bar(name='Output', x=top_tables['table_name'], y=top_tables['rows_output']),
    row=1, col=1
)

# 2. Taxa de duplicatas
metrics_df['dup_rate'] = (metrics_df['rows_duplicates'] / metrics_df['rows_input'] * 100).fillna(0)
top_dups = metrics_df.nlargest(10, 'dup_rate')
fig.add_trace(
    go.Bar(x=top_dups['table_name'], y=top_dups['dup_rate'], name='Dup %'),
    row=1, col=2
)

# 3. Quality Score
fig.add_trace(
    go.Histogram(x=metrics_df['quality_score'], name='Quality Score'),
    row=2, col=1
)

# 4. Throughput
top_throughput = metrics_df.nlargest(10, 'throughput_rows_per_sec')
fig.add_trace(
    go.Bar(x=top_throughput['table_name'], y=top_throughput['throughput_rows_per_sec'], name='Throughput'),
    row=2, col=2
)

fig.update_layout(
    height=800,
    title_text="Silver Layer Pipeline - Dashboard de Métricas",
    showlegend=True
)

fig.update_xaxes(tickangle=45)

# Salvar dashboard
dashboard_path = METRICS_DIR / f"dashboard_{datetime.now():%Y%m%d_%H%M%S}.html"
fig.write_html(str(dashboard_path))

logger.info(f"📊 Dashboard salvo: {dashboard_path}")

fig.show()

2026-02-08 00:22:53 | INFO     | __main__:<module> | 📊 Dashboard salvo: /app/output/metrics/silver/dashboard_20260208_002253.html


---
## 13. 📈 Relatório de Data Quality

In [29]:
# Analisar arquivos de quality reports
quality_reports = []

for report_file in QUALITY_DIR.glob("*_quality_report.json"):
    with open(report_file, 'r') as f:
        report = json.load(f)
        quality_reports.append(report)

if quality_reports:
    quality_df = pd.DataFrame(quality_reports)
    
    print("\n" + "="*80)
    print("✅ RELATÓRIO DE DATA QUALITY")
    print("="*80)
    print(f"\nTabelas analisadas: {len(quality_df)}")
    print(f"Quality Score médio: {quality_df['quality_score'].mean():.2f}%")
    print(f"\nTop 5 melhores tabelas:")
    print(quality_df.nlargest(5, 'quality_score')[['table_name', 'quality_score', 'checks_passed']])
    
    print(f"\nTabelas com issues:")
    issues_df = quality_df[quality_df['checks_failed'] > 0]
    if not issues_df.empty:
        print(issues_df[['table_name', 'checks_failed', 'quality_score']])
    else:
        print("✅ Nenhuma tabela com falhas!")
    
    print("="*80)
else:
    logger.warning("⚠️  Nenhum relatório de qualidade encontrado")


✅ RELATÓRIO DE DATA QUALITY

Tabelas analisadas: 3
Quality Score médio: 0.00%

Top 5 melhores tabelas:
       table_name  quality_score  checks_passed
0  depara_cliente            0.0              0
1          sc5030            0.0              0
2          sc6030            0.0              0

Tabelas com issues:
       table_name  checks_failed  quality_score
0  depara_cliente              2            0.0
1          sc5030              2            0.0
2          sc6030              2            0.0


---
## 14. 🔍 Validação Final

In [30]:
# CELL 30 - Validação Final: Verificar Tabelas Silver e Métricas
logger.info("🔍 Verificando tabelas no schema Silver...")

print("\n" + "="*80)
print("🥈 TABELAS SILVER CRIADAS")
print("="*80)

# Verificar tabelas Silver
silver_tables = client.query_df(f"""
    SELECT 
        name as table_name,
        total_rows,
        total_bytes,
        formatReadableSize(total_bytes) as size
    FROM system.tables
    WHERE database = '{CH_DATABASE_SILVER}'
    AND name NOT IN ('quality_metrics', 'performance_metrics', 'observability_metrics')
    ORDER BY total_rows DESC
""")

print(f"\nSchema Silver: {CH_DATABASE_SILVER}")
print(f"Total de tabelas: {len(silver_tables)}")

if len(silver_tables) > 0:
    print(f"\nTabelas criadas:\n")
    print(silver_tables.to_string(index=False))
    
    # Estatísticas
    total_rows = silver_tables['total_rows'].sum()
    total_size = silver_tables['total_bytes'].sum()
    
    print(f"\n📊 ESTATÍSTICAS:")
    print(f"Total de linhas: {total_rows:,}")
    print(f"Tamanho total: {total_size / (1024**3):.2f} GB")
    print(f"Média por tabela: {total_rows / len(silver_tables):,.0f} linhas")
else:
    print("\n⚠️  Nenhuma tabela Silver encontrada.")

print("="*80)

# Verificar tabelas de métricas
print("\n" + "="*80)
print("📊 TABELAS DE MÉTRICAS")
print("="*80)

metrics_tables = client.query_df(f"""
    SELECT 
        name as table_name,
        total_rows,
        formatReadableSize(total_bytes) as size
    FROM system.tables
    WHERE database = '{CH_DATABASE_SILVER}'
    AND name IN ('quality_metrics', 'performance_metrics', 'observability_metrics')
    ORDER BY name
""")

if len(metrics_tables) > 0:
    print(f"\n{metrics_tables.to_string(index=False)}")
    
    # Mostrar exemplos das métricas
    print("\n" + "-"*80)
    print("📈 ÚLTIMAS MÉTRICAS DE QUALIDADE:")
    print("-"*80)
    quality_sample = client.query_df(f"""
        SELECT 
            table_name,
            execution_timestamp,
            total_rows_input,
            total_rows_output,
            rows_duplicates,
            quality_score,
            checks_passed,
            checks_failed
        FROM {CH_DATABASE_SILVER}.quality_metrics
        ORDER BY execution_timestamp DESC
        LIMIT 10
    """)
    if len(quality_sample) > 0:
        print(quality_sample.to_string(index=False))
    
    print("\n" + "-"*80)
    print("⚡ ÚLTIMAS MÉTRICAS DE PERFORMANCE:")
    print("-"*80)
    perf_sample = client.query_df(f"""
        SELECT 
            table_name,
            execution_timestamp,
            duration_seconds,
            rows_processed,
            round(throughput_rows_per_sec, 2) as throughput,
            status
        FROM {CH_DATABASE_SILVER}.performance_metrics
        ORDER BY execution_timestamp DESC
        LIMIT 10
    """)
    if len(perf_sample) > 0:
        print(perf_sample.to_string(index=False))

print("="*80)

2026-02-08 00:22:54 | INFO     | __main__:<module> | 🔍 Verificando tabelas no schema Silver...



🥈 TABELAS SILVER CRIADAS

Schema Silver: track_silver
Total de tabelas: 6

Tabelas criadas:

    table_name  total_rows  total_bytes       size
        sd2030       10000       881087 860.44 KiB
        sc5030        9999       620530 605.99 KiB
        sc6030        9999       635253 620.36 KiB
 tst_contratos        9999       670822 655.10 KiB
        sf2030        9984       856531 836.46 KiB
depara_cliente          46         2911   2.84 KiB

📊 ESTATÍSTICAS:
Total de linhas: 50,027
Tamanho total: 0.00 GB
Média por tabela: 8,338 linhas

📊 TABELAS DE MÉTRICAS

           table_name  total_rows     size
observability_metrics          34 3.51 KiB
  performance_metrics           6 1.62 KiB
      quality_metrics           7 3.49 KiB

--------------------------------------------------------------------------------
📈 ÚLTIMAS MÉTRICAS DE QUALIDADE:
--------------------------------------------------------------------------------
    table_name execution_timestamp  total_rows_input  total_ro

---
## 15. 🎓 Conclusões e Próximos Passos

### ✅ O que foi implementado:

1. **Arquitetura Medallion - Camada Silver**
   - Limpeza e padronização de dados
   - Remoção de duplicatas
   - Tratamento de valores nulos

2. **Data Quality Framework**
   - Validações automáticas (completude, duplicatas, tipos)
   - Quality score por tabela
   - Relatórios detalhados

3. **Observabilidade**
   - Logging estruturado (JSON)
   - Métricas de performance
   - Dashboard interativo

4. **Performance**
   - Processamento paralelo com ThreadPool
   - Otimizações Spark (Adaptive Query Execution)
   - Caching estratégico

### 🚀 Próximos Passos:

1. **Camada Gold**
   - Criar agregações e modelos dimensionais
   - Implementar SCD (Slowly Changing Dimensions)
   - Criar views otimizadas para BI

2. **Orquestração**
   - Integrar com Airflow/Prefect
   - Scheduling automático
   - Retry logic e alertas

3. **Monitoramento Avançado**
   - Integração com Prometheus/Grafana
   - Alertas de qualidade de dados
   - SLA tracking

4. **CI/CD**
   - Testes automatizados de data quality
   - Deploy automatizado
   - Rollback strategy

### 📚 Referências:

- [Medallion Architecture](https://www.databricks.com/glossary/medallion-architecture)
- [Great Expectations](https://docs.greatexpectations.io/)
- [Spark Performance Tuning](https://spark.apache.org/docs/latest/sql-performance-tuning.html)
- [ClickHouse Best Practices](https://clickhouse.com/docs/en/guides/best-practices/)